# The Price is Right

## Order of Play

1: Modal.com and SpecialistAgent  
2: RAG, FrontierAgent, Ensemble Agent  
3: ScannerAgent, MessengerAgent  
4: AutonomousPlannerAgent and DealAgentFramework  
5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [01:17<00:00, 26.00s/it]


In [3]:
len(deals)

30

In [4]:
deals[3].describe()

"Title: Amazon Lowest Price in 365 Days Deals: Up to 57% off + free shipping\nDetails: At Amazon, you'll find deals that are at the lowest prices they've been in over a year here. It's a great place to shop since Prime Day is over and lots of prices have increased across the store. You can be guaranteed that the deals in here are at the best prices they've listed in a year. Shop for savings across phones, electronics, lawn care essentials, apparel basics, and more. Shipping is free for Prime members.\xa0 Shop Now at Amazon\nFeature: \nURL: https://www.dealnews.com/Amazon-Lowest-Price-in-365-Days-Deals-Up-to-57-off-free-shipping/21858076.html?iref=rss-c142"

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Amazon College Tech Deals for Up to 50% off, deals from $30 + free shipping w/ Prime
Details: Amazon's College Tech Deals event includes laptops, headphones, speakers, gaming accessories, and more, with deals starting at $30 and discounts up to 50% off. Prime members get free shipping on eligible orders, too. There are brands big and small in this sale, including Sony, Apple, X

In [8]:
DealSelection.model_json_schema()

{'$defs': {'Deal': {'description': 'A class to Represent a Deal with a summary description',
   'properties': {'product_description': {'description': "Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose.",
     'title': 'Product Description',
     'type': 'string'},
    'price': {'description': 'The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200',
     'title': 'Price',
     'type': 'number'},
    'url': {'description': 'The URL of the deal, as provided in the input',
     'title': 'Url',
     'type': 'string'}},
   'required': ['product_description', 'price', 'url'],
   'title': 'Deal',
   'type': 'object'}},
 'description': 'A clas

In [9]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Igloo Kool Tunes 14-Quart Retro Bluetooth Boombox Cooler combines a 14-quart insulated cooler that fits up to 26 standard 12‑oz cans with integrated dual 5W external speakers and passive radiators. It uses Bluetooth 5.0 for wireless pairing, supports KoolSync multi-unit stereo pairing, and has an IP56 rating for water and dust resistance. The rechargeable battery provides up to 10 hours of audio playtime per charge while preserving full interior storage since the speakers are mounted externally.', price=70.0, url='https://www.dealnews.com/Igloo-Kool-Tunes-14-Quart-Retro-Bluetooth-Boombox-Cooler-for-70-free-shipping/21858079.html?iref=rss-c142'), Deal(product_description='Refurbished Unlocked Samsung Galaxy S24 Ultra (model S928U) offers a 6.8-inch display, 12GB of RAM and 512GB of onboard storage, and a 200MP rear camera driven by an octa-core processor. The listing is for a certified refurbished unit noted as fully functional but may show

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='Igloo Kool Tunes 14-Quart Retro Bluetooth Boombox Cooler combines a 14‑quart insulated cooler that fits up to 26 standard 12‑oz cans with integrated external stereo audio. It features dual 5W speakers with passive radiators, Bluetooth 5.0 pairing, KoolSync multi‑unit stereo pairing, an 80Hz–20kHz frequency response, IP56 water‑ and dust‑resistance, and up to 10 hours of playback per charge — preserving full internal storage since the speakers are mounted externally.', price=70.0, url='https://www.dealnews.com/Igloo-Kool-Tunes-14-Quart-Retro-Bluetooth-Boombox-Cooler-for-70-free-shipping/21858079.html?iref=rss-c142'), Deal(product_description='Refurbished Unlocked Samsung Galaxy S24 Ultra with 512GB storage is a high‑end flagship phone featuring a 6.8" display, 12GB of RAM, a 200MP rear camera array, and an octa‑core processor. This unlocked S928U model offers large onboard storage for photos and apps and is sold refurbished in “Good” condit

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [14]:
load_dotenv(override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [1]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [2]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")